In [3]:
import pandas as pd

PATH_APLICACAO = "fam_pes_cad_11_25_amostra_aplicacao.csv"
df_app = pd.read_csv(PATH_APLICACAO)

print(df_app.shape)
df_app.head()

(983850, 47)


,ID_FAM_ANON,VL_RENDA_MEDIA_FAM,IN_TRABALHO_INFANTIL_FAM,CO_MUNIC_IBGE_2_FAM,CO_MUNIC_IBGE_5_FAM,QT_PESSOAS_DOMIC_FAM,QT_FAMILIAS_DOMIC_FAM,CO_ESPECIE_DOMIC_FAM,CO_LOCAL_DOMIC_FAM,QT_COMODOS_DOMIC_FAM,...,PCT_ADULTOS_30A59,PCT_IDOSOS_60A64,PCT_IDOSOS_BPC,PCT_PES_DEFICIENCIA,TEM_CRIANCA_SEM_ESCOLA,TEM_ADOLESCENTE_SEM_ESCOLA,PCT_PES_ANALFABETA,PCT_ADULTO_NUNCA_FREQ_ESCOLA,PCT_7A18_ESCOLA_PUBLICA,PCT_MENOR6_FORA_CRECHE_PRE
0,182936,0.0,2,31,48509,1,1,2,1,-1,...,1.000000,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0
1,69409,200.0,2,33,5901,3,1,1,2,6,...,0.666667,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0
2,835658,1518.0,2,28,308,2,1,1,1,6,...,0.000000,0.0,1.0,0.0,0,0,1.0,1.0,0.0,0.0
3,823013,2118.0,2,52,6305,1,1,1,1,5,...,1.000000,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0
4,636430,180.0,2,35,50308,1,1,1,1,3,...,0.000000,1.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0


In [4]:
from models.xgb_threshold import XGBoostComThreshold


In [5]:
import joblib
from pathlib import Path

PATH_MODEL = Path("models") / "pipeline_xgb_binario_thr080.joblib"

pipeline = joblib.load(PATH_MODEL)

pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('quant',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   MinMaxScaler())]),
                                                  ['QT_PESSOAS_DOMIC_FAM',
                                                   'QT_FAMILIAS_DOMIC_FAM',
                                                   'QT_COMODOS_DOMIC_FAM',
                                                   'QT_COMODOS_DORMITORIO_FAM',
                                                   'QTD_PESSOAS',
                                                   'IDADE_REFERENCIA']),
                                                 ('pct', 'passthrough',
                                                  ['PCT_1_INFANCIA',
                                                   'PCT_CRIANCAS_7A11'...
                                                   'CO_ESCOA_SANITARIO_DOMIC_FAM',
                                                   'CO_ILUMINACAO_DOMIC_FAM',
                                                   'IN_PARC_MDS_FAM',
                                                   'CO_RACA_COR_PESSOA',
                                                   'IN_FREQUENTA_ESCOLA_MEMB',
                                                   'CO_CURSO_FREQUENTA_MEMB',
                                                   'CO_CURSO_FREQ_PESSOA_MEMB',
                                                   'CO_AFASTADO_TRAB_MEMB',
                                                   'CO_PRINCIPAL_TRAB_MEMB',
                                                   'CO_TRABALHO_12_MESES_MEMB']),
                                                 ('geo', 'passthrough',
                                                  ['CO_MUNIC_IBGE_2_FAM',
                                                   'CO_MUNIC_IBGE_5_FAM'])])),
                ('model', XGBoostComThreshold())])

In [6]:
# Probabilidade da classe positiva (renda informal)
proba = pipeline.predict_proba(df_app)[:, 1]

# Predição final (já com threshold 0.80)
pred = pipeline.predict(df_app)


In [7]:
df_result = df_app.copy()

df_result["PROB_RENDA_INFORMAL"] = proba
df_result["PRED_RENDA_INFORMAL"] = pred

df_result.head()


,ID_FAM_ANON,VL_RENDA_MEDIA_FAM,IN_TRABALHO_INFANTIL_FAM,CO_MUNIC_IBGE_2_FAM,CO_MUNIC_IBGE_5_FAM,QT_PESSOAS_DOMIC_FAM,QT_FAMILIAS_DOMIC_FAM,CO_ESPECIE_DOMIC_FAM,CO_LOCAL_DOMIC_FAM,QT_COMODOS_DOMIC_FAM,...,PCT_IDOSOS_BPC,PCT_PES_DEFICIENCIA,TEM_CRIANCA_SEM_ESCOLA,TEM_ADOLESCENTE_SEM_ESCOLA,PCT_PES_ANALFABETA,PCT_ADULTO_NUNCA_FREQ_ESCOLA,PCT_7A18_ESCOLA_PUBLICA,PCT_MENOR6_FORA_CRECHE_PRE,PROB_RENDA_INFORMAL,PRED_RENDA_INFORMAL
0,182936,0.0,2,31,48509,1,1,2,1,-1,...,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.125069,0
1,69409,200.0,2,33,5901,3,1,1,2,6,...,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.545528,0
2,835658,1518.0,2,28,308,2,1,1,1,6,...,1.0,0.0,0,0,1.0,1.0,0.0,0.0,0.990929,1
3,823013,2118.0,2,52,6305,1,1,1,1,5,...,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.230321,0
4,636430,180.0,2,35,50308,1,1,1,1,3,...,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.572513,0


In [8]:
# selecionar apenas as colunas desejadas
cols_saida = [
    "ID_FAM_ANON",
    "VL_RENDA_MEDIA_FAM",
    "CO_MUNIC_IBGE_2_FAM",
    "CO_MUNIC_IBGE_5_FAM",
    "PROB_RENDA_INFORMAL",
    "PRED_RENDA_INFORMAL",
]

df_saida = df_result[cols_saida].copy()

# salvar arquivo final
df_saida.to_csv(
    "familias_ml_resumo.csv",
    index=False
)

print("✅ Arquivo salvo: familias_ml_resumo.csv")


✅ Arquivo salvo: familias_ml_resumo.csv
